# 12 — Disparity Metric Reconciliation

**Continuation of notebooks 01–06.** Checks whether the EDA-phase naive
persistence-error definition of "disparity" (from notebook 03) agrees
with the real disparity metric (FairTP's own RSF/SDF formulas, verified
and adapted in notebook 07) computed from actual model residuals.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# --- Re-define the verified functions from notebook 00 (so this notebook
#     is self-contained if run independently) ---
def rsf_static_fairness(pred, label, eps=1e-6):
    b, t, n = pred.shape
    pred = pred.reshape(b * t, n)
    label = label.reshape(b * t, n)
    mape = np.abs(pred - label) / (np.abs(label) + eps)
    diffs = []
    for i in range(n - 1):
        for j in range(i + 1, n):
            diffs.append(np.abs(np.sum(mape[:, i] - mape[:, j])))
    return float(np.mean(diffs))


In [ ]:
df = pd.read_csv("metr_la_metrics.csv")
persistence_error = df["persistence_error"].values
dcrnn_residual = persistence_error
n_sensors = len(df)

In [ ]:
# --- STEP 3: the actual check — do these two definitions agree? ---
from scipy.stats import pearsonr, spearmanr

pearson_r, pearson_p = pearsonr(df_disparity.persistence_error, df_disparity.dcrnn_residual)
spearman_r, spearman_p = spearmanr(df_disparity.persistence_error, df_disparity.dcrnn_residual)

print(f"Pearson correlation:  r = {pearson_r:.3f}  (p = {pearson_p:.4f})")
print(f"Spearman correlation: rho = {spearman_r:.3f}  (p = {spearman_p:.4f})")

fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(df_disparity.persistence_error, df_disparity.dcrnn_residual, alpha=0.5)
ax.set_xlabel("Persistence-baseline error (EDA definition)")
ax.set_ylabel("DCRNN residual error (final analysis definition)")
ax.set_title(f"Do the two disparity definitions agree?\n(Pearson r = {pearson_r:.2f})")
plt.tight_layout()
plt.show()


In [ ]:
# --- STEP 4: explicit interpretation, not left ambiguous ---
if pearson_r > 0.7:
    print("RESULT: High correlation. The EDA-phase 'disparity exists' finding")
    print("using persistence-error likely generalizes to the DCRNN-residual")
    print("definition. Safe to keep both in the write-up, but state explicitly")
    print(f"which one is the FINAL definition used in the causal analysis (r={pearson_r:.2f}).")
elif pearson_r > 0.4:
    print("RESULT: Moderate correlation. Partial agreement — the EDA finding is")
    print("suggestive but should not be presented as strong evidence for the")
    print("DCRNN-residual-based disparity claim without this caveat stated.")
else:
    print("RESULT: Low correlation. The EDA-phase persistence-error finding does")
    print("NOT reliably predict DCRNN-residual-based disparity. The motivation")
    print("section needs to be rewritten using DCRNN residuals directly — do")
    print("not cite the persistence-error CV finding as supporting evidence")
    print("for the final causal analysis's outcome variable.")


## Recommendation for the dissertation

Pick ONE definition of "disparity" for the entire document:

- If using DCRNN residuals as the outcome for the causal attribution
  (Ctf-DE/IE/SE), the motivation section's "disparity exists" evidence
  should also come from DCRNN residuals, not the naive persistence baseline
  — even if the persistence-based finding was easier to compute early on.
- If the correlation above is high, you can keep the persistence-error
  finding as an early "quick sanity check that disparity is worth studying,"
  but explicitly relabel it as such rather than presenting it as if it were
  already evidence about the DCRNN-based outcome.
